# The chat store: SQLAlchemy ORM, the writing half

Notebook 01 only read through the ORM. Here it writes: models with a relationship, sessions that add and commit, and a cascading delete.

The chats live in their own database (`text2sql_app`), written by `app_writer`. The agent keeps its read-only engine on Chinook. Neither login can open the other's database.

Setup SQL is in the README, step 6. Run that first.

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

load_dotenv()

def pg_url(user, password, database):
    return URL.create("postgresql+psycopg2", username=user, password=password,
                      host="localhost", port=5432, database=database)

app_url = pg_url(os.environ["APP_DB_USER"], os.environ["APP_DB_PASSWORD"], os.environ["APP_DB_NAME"])
app_engine = create_engine(app_url)

with app_engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_database()")).one())

KeyError: 'APP_DB_USER'

## Prove the wall between the two

Same passwords, wrong database. Both should fail with `permission denied for database`, because README step 6 revoked `CONNECT` from `PUBLIC` on both.

In [ ]:
wrong_doors = {
    "app_writer -> Chinook": pg_url(os.environ["APP_DB_USER"], os.environ["APP_DB_PASSWORD"], os.environ["POSTGRES_DB"]),
    "readonly_user -> chats": pg_url(os.environ["READONLY_DB_USER"], os.environ["READONLY_DB_PASSWORD"], os.environ["APP_DB_NAME"]),
}
for label, url in wrong_doors.items():
    try:
        with create_engine(url).connect():
            print(label, "CONNECTED, the wall is missing")
    except Exception as e:
        print(label, "->", type(e).__name__, str(e).splitlines()[0][-60:])

## Two models and a relationship

`Chat` has many `Message`s. `relationship()` is Python-side only. It gives `chat.messages` and `message.chat`, and the `ForeignKey` is what Postgres enforces.

`cascade="all, delete-orphan"` means deleting a chat through the session deletes its messages too.

In [ ]:
import uuid
from datetime import datetime
from sqlalchemy import JSON, DateTime, ForeignKey, String, Text, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

class Base(DeclarativeBase):
    pass

class Chat(Base):
    __tablename__ = "chats"
    id: Mapped[uuid.UUID] = mapped_column(primary_key=True, default=uuid.uuid4)
    title: Mapped[str] = mapped_column(String(60), default="New chat")
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), server_default=func.now())
    updated_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), server_default=func.now(), onupdate=func.now())
    messages: Mapped[list["Message"]] = relationship(back_populates="chat", cascade="all, delete-orphan", order_by="Message.id")

class Message(Base):
    __tablename__ = "messages"
    id: Mapped[int] = mapped_column(primary_key=True)
    chat_id: Mapped[uuid.UUID] = mapped_column(ForeignKey("chats.id", ondelete="CASCADE"), index=True)
    role: Mapped[str] = mapped_column(String(16))
    content: Mapped[str] = mapped_column(Text)
    sql: Mapped[str | None] = mapped_column(Text)
    rows: Mapped[list | None] = mapped_column(JSON)
    status: Mapped[str | None] = mapped_column(String(16))
    attempts: Mapped[int | None]
    events: Mapped[list | None] = mapped_column(JSON)
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), server_default=func.now())
    chat: Mapped[Chat] = relationship(back_populates="messages")

In [ ]:
# echo on to see the DDL create_all sends. It skips tables that already exist.
app_engine.echo = True
Base.metadata.create_all(app_engine)
app_engine.echo = False

## Writing: add, flush, commit

`session.add()` only stages the object. `flush()` sends the INSERT inside the open transaction, which is when defaults like the uuid get filled in. `commit()` makes it permanent. Nothing is saved until the commit.

In [ ]:
from sqlalchemy.orm import Session

with Session(app_engine) as session:
    chat = Chat(title="How many tracks?")
    chat.messages.append(Message(role="user", content="How many tracks are in the database?"))
    chat.messages.append(Message(role="assistant", content="Query returned 1 row(s).",
                                 sql="SELECT COUNT(*) FROM track", rows=[{"count": 3503}],
                                 status="ok", attempts=0, events=[]))
    session.add(chat)
    print("before flush:", chat.id, [m.id for m in chat.messages])
    session.flush()
    print("after flush: ", chat.id, [m.id for m in chat.messages])
    session.commit()
    chat_id = chat.id

## Reading back, and lazy loading

`chat.messages` isn't loaded when the chat is. The SELECT for messages fires the first time you touch the attribute. Watch the echo output.

In [ ]:
from sqlalchemy import select

with Session(app_engine) as session:
    app_engine.echo = True
    chat = session.get(Chat, chat_id)
    print("--- chat loaded, touching .messages now ---")
    for m in chat.messages:
        print(m.role, "|", m.content, "|", m.rows)
    app_engine.echo = False

    # the sidebar query: newest first
    for c in session.scalars(select(Chat).order_by(Chat.updated_at.desc()).limit(5)):
        print(c.updated_at, c.title)

## Delete cascades

Deleting the chat through the session deletes its messages too.

In [ ]:
from sqlalchemy import func as sa_func

with Session(app_engine) as session:
    session.delete(session.get(Chat, chat_id))
    session.commit()
    left = session.scalar(select(sa_func.count()).select_from(Message).where(Message.chat_id == chat_id))
    print("messages left for that chat:", left)